# 03h — Code-Point Open: Master Postcode Lookup

**Purpose:** Build and validate the master postcode → BNG coordinate lookup from OS Code-Point Open (Feb 2026 release).
This lookup enables geocoding of NHS ODS hospitals, GP surgeries, and GIAS schools that carry postcodes
but lack explicit coordinates. All coordinates are British National Grid (BNG, EPSG:27700); WGS84 conversion
is available on demand via pyproj.

**Input:** `data/raw/boundaries/code_point_open/data/CSV/` — 120 CSV files, no header row
**Output:** `data/audit/postcode_lookup.parquet`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from loguru import logger

ROOT = Path("..") if Path("../data").exists() else Path(".")
CSV_DIR = ROOT / "data/raw/boundaries/code_point_open/data/CSV"
AUDIT_DIR = ROOT / "data/audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

COLS = ["Postcode", "PQI", "Easting", "Northing", "Country", "NHS_HA", "NHS_RHA", "County", "District", "Ward"]

logger.info(f"Code-Point CSV dir: {CSV_DIR.resolve()}")

2026-03-13 00:08:00.925 | INFO     | __main__:<module>:13 - Code-Point CSV dir: /Users/souravamseekarmarti/Projects/aequitas/data/raw/boundaries/code_point_open/data/CSV


## Section 1 — Discovery

In [2]:
csv_files = sorted(CSV_DIR.glob("*.csv"))
logger.info(f"Total CSV files found: {len(csv_files)}")
print(f"Total CSV files: {len(csv_files)}")
print(f"First 10: {[f.name for f in csv_files[:10]]}")
print(f"Last 10:  {[f.name for f in csv_files[-10:]]}")

# Load one sample file to verify structure
sample = pd.read_csv(csv_files[0], header=None, names=COLS)
print(f"\nSample file: {csv_files[0].name} — {len(sample):,} rows")
print(sample.head(5).to_string())
print(f"\nDtypes:\n{sample.dtypes}")

2026-03-13 00:08:00.930 | INFO     | __main__:<module>:2 - Total CSV files found: 120


Total CSV files: 120
First 10: ['ab.csv', 'al.csv', 'b.csv', 'ba.csv', 'bb.csv', 'bd.csv', 'bh.csv', 'bl.csv', 'bn.csv', 'br.csv']
Last 10:  ['wa.csv', 'wc.csv', 'wd.csv', 'wf.csv', 'wn.csv', 'wr.csv', 'ws.csv', 'wv.csv', 'yo.csv', 'ze.csv']

Sample file: ab.csv — 17,386 rows
   Postcode  PQI  Easting  Northing    Country  NHS_HA    NHS_RHA  County   District       Ward
0  AB10 1AB   10   394235    806529  S92000003     NaN  S08000020     NaN  S12000033  S13002842
1  AB10 1AF   10   394235    806529  S92000003     NaN  S08000020     NaN  S12000033  S13002842
2  AB10 1AG   10   394230    806469  S92000003     NaN  S08000020     NaN  S12000033  S13002842
3  AB10 1AH   10   394235    806529  S92000003     NaN  S08000020     NaN  S12000033  S13002842
4  AB10 1AL   10   394296    806581  S92000003     NaN  S08000020     NaN  S12000033  S13002842

Dtypes:
Postcode        str
PQI           int64
Easting       int64
Northing      int64
Country         str
NHS_HA      float64
NHS_RHA         st

## Section 2 — Load All 120 Files

In [3]:
dfs = []
for f in csv_files:
    df = pd.read_csv(f, header=None, names=COLS, dtype={"Postcode": str})
    dfs.append(df)

full = pd.concat(dfs, ignore_index=True)
logger.info(f"Full dataset loaded: {len(full):,} rows")
print(f"Total rows (all countries): {len(full):,}")
print(f"Memory usage: {full.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nDtypes:\n{full.dtypes}")
print(f"\nCountry value counts:\n{full['Country'].value_counts()}")

2026-03-13 00:08:02.556 | INFO     | __main__:<module>:7 - Full dataset loaded: 1,746,551 rows


Total rows (all countries): 1,746,551


Memory usage: 353.0 MB

Dtypes:
Postcode       str
PQI          int64
Easting      int64
Northing     int64
Country        str
NHS_HA      object
NHS_RHA        str
County      object
District       str
Ward           str
dtype: object

Country value counts:
Country
E92000001    1492853
S92000003     161573
W92000004      92125
Name: count, dtype: int64


## Section 3 — Null Checks

In [4]:
null_counts = full.isnull().sum()
print("Null counts per column:")
print(null_counts.to_string())

assert null_counts["Postcode"] == 0, "FAIL: Postcode has nulls"
assert null_counts["Easting"] == 0, "FAIL: Easting has nulls"
assert null_counts["Northing"] == 0, "FAIL: Northing has nulls"
logger.info("PASS: Postcode, Easting, Northing have zero nulls")
print("\nPASS: Postcode, Easting, Northing — zero nulls confirmed")

2026-03-13 00:08:02.985 | INFO     | __main__:<module>:8 - PASS: Postcode, Easting, Northing have zero nulls


Null counts per column:
Postcode          0
PQI               0
Easting           0
Northing          0
Country           0
NHS_HA       254734
NHS_RHA        1068
County      1188991
District       1068
Ward           1068

PASS: Postcode, Easting, Northing — zero nulls confirmed


## Section 4 — Filter to England

In [5]:
# Country code E92000001 = England (confirmed from sample exploration)
england = full[full["Country"] == "E92000001"].copy()
logger.info(f"England rows: {len(england):,}")
print(f"England rows (Country == E92000001): {len(england):,}")
print(f"Non-England rows excluded: {len(full) - len(england):,}")

assert len(england) > 1_400_000, f"FAIL: England rows {len(england):,} < 1,400,000"
logger.info(f"PASS: England postcode count > 1,400,000 (actual: {len(england):,})")
print(f"\nPASS: England postcodes > 1,400,000 — actual: {len(england):,}")

2026-03-13 00:08:03.138 | INFO     | __main__:<module>:3 - England rows: 1,492,853


2026-03-13 00:08:03.138 | INFO     | __main__:<module>:8 - PASS: England postcode count > 1,400,000 (actual: 1,492,853)


England rows (Country == E92000001): 1,492,853
Non-England rows excluded: 253,698

PASS: England postcodes > 1,400,000 — actual: 1,492,853


## Section 5 — Postcode Standardisation

In [6]:
# Strip whitespace
england["Postcode"] = england["Postcode"].str.strip()

# Standardise to "OUTWARD INWARD" format with exactly one space before last 3 chars
def standardise_postcode(pc: str) -> str:
    """Ensure postcode has single space before last 3 characters (inward code)."""
    pc = pc.strip().upper().replace(" ", "")
    if len(pc) >= 5:
        return pc[:-3] + " " + pc[-3:]
    return pc

england["Postcode"] = england["Postcode"].apply(standardise_postcode)

# Check for duplicates
dup_count = england["Postcode"].duplicated().sum()
logger.info(f"Duplicate postcodes: {dup_count}")
print(f"Duplicate postcodes: {dup_count}")
if dup_count > 0:
    logger.warning(f"WARN: {dup_count} duplicate postcodes found — keeping first occurrence")
    print(f"WARN: {dup_count} duplicates found — deduplicating by keeping first")
    england = england.drop_duplicates(subset="Postcode", keep="first")
    print(f"Rows after dedup: {len(england):,}")
else:
    print("PASS: No duplicate postcodes")

print(f"\nSample standardised postcodes:")
print(england["Postcode"].head(10).tolist())

2026-03-13 00:08:03.808 | INFO     | __main__:<module>:16 - Duplicate postcodes: 0


Duplicate postcodes: 0
PASS: No duplicate postcodes

Sample standardised postcodes:
['AL1 1AG', 'AL1 1AJ', 'AL1 1AR', 'AL1 1AS', 'AL1 1AT', 'AL1 1AU', 'AL1 1AW', 'AL1 1AX', 'AL1 1BH', 'AL1 1BX']


## Section 6 — Coordinate Validation

In [7]:
# Valid England BNG ranges
E_MIN, E_MAX = 80_000, 660_000
N_MIN, N_MAX = 5_000, 700_000

# Identify non-geographic postcodes (PO boxes, large organisations): Easting=0, Northing=0
# These are a known feature of Code-Point Open; they carry no spatial information
non_geo = england[(england["Easting"] == 0) & (england["Northing"] == 0)]
logger.info(f"Non-geographic postcodes (E=0,N=0) excluded: {len(non_geo)}")
print(f"Non-geographic postcodes (Easting=0, Northing=0): {len(non_geo)} — excluded from lookup")
if len(non_geo) > 0:
    print(f"  Sample: {non_geo['Postcode'].head(10).tolist()}")

# Remove non-geographic rows before range check
england = england[(england["Easting"] > 0) & (england["Northing"] > 0)].copy()
logger.info(f"England postcodes after removing non-geographic: {len(england):,}")

e_oor = england[(england["Easting"] < E_MIN) | (england["Easting"] > E_MAX)]
n_oor = england[(england["Northing"] < N_MIN) | (england["Northing"] > N_MAX)]

logger.info(f"Easting out-of-range (after non-geo removal): {len(e_oor)}")
logger.info(f"Northing out-of-range (after non-geo removal): {len(n_oor)}")
print(f"\nEasting out-of-range  (< {E_MIN:,} or > {E_MAX:,}): {len(e_oor)}")
print(f"Northing out-of-range (< {N_MIN:,} or > {N_MAX:,}): {len(n_oor)}")

if len(e_oor) > 0:
    print(f"  Sample Easting outliers: {e_oor[['Postcode','Easting','Northing']].head(5).to_string()}")
if len(n_oor) > 0:
    print(f"  Sample Northing outliers: {n_oor[['Postcode','Easting','Northing']].head(5).to_string()}")

print(f"\nEasting  — min: {england['Easting'].min():,}  max: {england['Easting'].max():,}  mean: {england['Easting'].mean():,.0f}")
print(f"Northing — min: {england['Northing'].min():,}  max: {england['Northing'].max():,}  mean: {england['Northing'].mean():,.0f}")

2026-03-13 00:08:03.817 | INFO     | __main__:<module>:8 - Non-geographic postcodes (E=0,N=0) excluded: 837


2026-03-13 00:08:03.913 | INFO     | __main__:<module>:15 - England postcodes after removing non-geographic: 1,492,016


2026-03-13 00:08:03.918 | INFO     | __main__:<module>:20 - Easting out-of-range (after non-geo removal): 0


2026-03-13 00:08:03.918 | INFO     | __main__:<module>:21 - Northing out-of-range (after non-geo removal): 0


Non-geographic postcodes (Easting=0, Northing=0): 837 — excluded from lookup
  Sample: ['BD98 1GA', 'BD98 1GB', 'BD98 1GD', 'BD98 1GG', 'BD98 1YA', 'BD98 1YB', 'BD98 1YD', 'BD98 1YY', 'BD98 5GA', 'BD98 5GG']

Easting out-of-range  (< 80,000 or > 660,000): 0
Northing out-of-range (< 5,000 or > 700,000): 0

Easting  — min: 87,897  max: 655,448  mean: 451,386
Northing — min: 8,478  max: 656,014  mean: 275,258


## Section 7 — BNG → WGS84 Validation (Sample of 1000)

In [8]:
from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)

sample_1000 = england.sample(1000, random_state=42)
lons, lats = transformer.transform(
    sample_1000["Easting"].values,
    sample_1000["Northing"].values,
)

lat_arr = np.array(lats)
lon_arr = np.array(lons)

# Slightly relaxed bounds: Isles of Scilly reach ~49.85, Cornwall SW tip ~-5.7,
# Northumberland reaches ~55.8 — using generous England envelope
lat_ok = np.all((lat_arr >= 49.5) & (lat_arr <= 56.0))
lon_ok = np.all((lon_arr >= -8.0) & (lon_arr <= 2.0))

print(f"WGS84 validation (n=1000 sample):")
print(f"  Lat range: {lat_arr.min():.4f} – {lat_arr.max():.4f}  (expected ~49.5–56.0 England envelope)  → {'PASS' if lat_ok else 'FAIL'}")
print(f"  Lon range: {lon_arr.min():.4f} – {lon_arr.max():.4f}  (expected ~-8.0–2.0 England envelope)  → {'PASS' if lon_ok else 'FAIL'}")

assert lat_ok, f"FAIL: Latitudes out of England range — min={lat_arr.min():.4f}, max={lat_arr.max():.4f}"
assert lon_ok, f"FAIL: Longitudes out of England range — min={lon_arr.min():.4f}, max={lon_arr.max():.4f}"
logger.info("PASS: BNG→WGS84 sample validation passed")

2026-03-13 00:08:04.009 | INFO     | __main__:<module>:25 - PASS: BNG→WGS84 sample validation passed


WGS84 validation (n=1000 sample):
  Lat range: 50.0282 – 55.2575  (expected ~49.5–56.0 England envelope)  → PASS
  Lon range: -5.6694 – 1.7377  (expected ~-8.0–2.0 England envelope)  → PASS


## Section 8 — Save Output

In [9]:
output = england[["Postcode", "Easting", "Northing"]].copy()
output["Easting"] = output["Easting"].astype(int)
output["Northing"] = output["Northing"].astype(int)

out_path = AUDIT_DIR / "postcode_lookup.parquet"
output.to_parquet(out_path, index=False)
logger.info(f"Saved postcode_lookup.parquet: {len(output):,} rows → {out_path}")
print(f"\nSaved: {out_path}")
print(f"Final row count: {len(output):,}")
print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")
print(output.head(5).to_string())

2026-03-13 00:08:04.154 | INFO     | __main__:<module>:7 - Saved postcode_lookup.parquet: 1,492,016 rows → ../data/audit/postcode_lookup.parquet



Saved: ../data/audit/postcode_lookup.parquet
Final row count: 1,492,016
File size: 18.1 MB
      Postcode  Easting  Northing
17386  AL1 1AG   515487    206498
17387  AL1 1AJ   515491    206410
17388  AL1 1AR   516270    205897
17389  AL1 1AS   515005    206908
17390  AL1 1AT   516131    206148


## Section 9 — Coverage Check (Hospitals & GPs)

In [10]:
# Load hospitals — file has 14 blank/junk lines before the real header
hospitals = pd.read_csv(
    ROOT / "data/raw/poi/hospitals.csv",
    skiprows=14,
    names=["OrgId", "Name", "PostCode", "LastChangeDate"],
    dtype=str,
)
# Remove any rows where OrgId looks like a header repeat or junk
hospitals = hospitals[hospitals["OrgId"].str.match(r'^[A-Z0-9]{4,6}$', na=False)].copy()
hospitals["PostCode"] = hospitals["PostCode"].str.strip().apply(
    lambda x: standardise_postcode(x) if pd.notna(x) else x
)
hospitals = hospitals.dropna(subset=["PostCode"])
logger.info(f"Hospitals for coverage check: {len(hospitals)}")

# Load GP surgeries
gp = pd.read_csv(ROOT / "data/raw/poi/epraccur.csv", dtype=str)
gp["PostCode"] = gp["PostCode"].str.strip().apply(
    lambda x: standardise_postcode(x) if pd.notna(x) else x
)
gp = gp.dropna(subset=["PostCode"])
logger.info(f"GP surgeries for coverage check: {len(gp)}")

lookup_set = set(output["Postcode"])

hosp_match = hospitals["PostCode"].isin(lookup_set).sum()
hosp_pct = 100 * hosp_match / len(hospitals)
gp_match = gp["PostCode"].isin(lookup_set).sum()
gp_pct = 100 * gp_match / len(gp)

print(f"Hospitals  — {hosp_match:,}/{len(hospitals):,} postcodes matched  ({hosp_pct:.2f}%)")
print(f"GP surgeries — {gp_match:,}/{len(gp):,} postcodes matched  ({gp_pct:.2f}%)")

# Log unmatched
hosp_unmatched = hospitals[~hospitals["PostCode"].isin(lookup_set)]["PostCode"].tolist()
gp_unmatched = gp[~gp["PostCode"].isin(lookup_set)]["PostCode"].tolist()
if hosp_unmatched:
    logger.warning(f"Unmatched hospital postcodes ({len(hosp_unmatched)}): {hosp_unmatched[:20]}")
    print(f"  Unmatched hospitals: {hosp_unmatched[:20]}")
if gp_unmatched:
    logger.warning(f"Unmatched GP postcodes ({len(gp_unmatched)}): {gp_unmatched[:20]}")
    print(f"  Unmatched GPs: {gp_unmatched[:20]}")

assert hosp_pct >= 95.0, f"FAIL: Hospital postcode coverage {hosp_pct:.2f}% < 95%"
assert gp_pct >= 95.0, f"FAIL: GP postcode coverage {gp_pct:.2f}% < 95%"
logger.info("PASS: Hospital and GP postcode coverage both >= 95%")

2026-03-13 00:08:04.169 | INFO     | __main__:<module>:14 - Hospitals for coverage check: 3870


2026-03-13 00:08:04.185 | INFO     | __main__:<module>:22 - GP surgeries for coverage check: 12213


Hospitals  — 3,714/3,870 postcodes matched  (95.97%)
GP surgeries — 12,059/12,213 postcodes matched  (98.74%)


2026-03-13 00:08:23.266 | WARNING  | __main__:<module>:38 - Unmatched hospital postcodes (156): ['CT17 0HD', 'RH1 6YY', 'BN21 3BG', 'CB2 0AU', 'SL5 8AA', 'PO4 8LD', 'SO43 7NG', 'SY3 8DN', 'TF8 7NZ', 'ST2 8LD', 'E2 9JX', 'BS16 1JE', 'BS35 1DN', 'TQ6 9BD', 'RM7 9BH', 'CM15 8DP', 'WC1X 8DA', 'W6 0TN', 'BA11 1EY', 'GL56 0BS']


2026-03-13 00:08:23.266 | WARNING  | __main__:<module>:41 - Unmatched GP postcodes (154): ['NE9 5UR', 'KT18 6JW', 'TW2 7DU', 'BF1 0AP', 'BF1 0DN', 'BF1 0AF', 'BF1 0AB', 'BF1 2AG', 'BF1 0AS', 'BF1 2AH', 'BF1 0DL', 'BF1 2AB', 'BF1 2AT', 'BF1 2AU', 'BF1 2AS', 'BF1 2AR', 'BF1 6FA', 'BF1 3AJ', 'BF1 6DU', 'BF1 3AG']


2026-03-13 00:08:23.267 | INFO     | __main__:<module>:46 - PASS: Hospital and GP postcode coverage both >= 95%


  Unmatched hospitals: ['CT17 0HD', 'RH1 6YY', 'BN21 3BG', 'CB2 0AU', 'SL5 8AA', 'PO4 8LD', 'SO43 7NG', 'SY3 8DN', 'TF8 7NZ', 'ST2 8LD', 'E2 9JX', 'BS16 1JE', 'BS35 1DN', 'TQ6 9BD', 'RM7 9BH', 'CM15 8DP', 'WC1X 8DA', 'W6 0TN', 'BA11 1EY', 'GL56 0BS']
  Unmatched GPs: ['NE9 5UR', 'KT18 6JW', 'TW2 7DU', 'BF1 0AP', 'BF1 0DN', 'BF1 0AF', 'BF1 0AB', 'BF1 2AG', 'BF1 0AS', 'BF1 2AH', 'BF1 0DL', 'BF1 2AB', 'BF1 2AT', 'BF1 2AU', 'BF1 2AS', 'BF1 2AR', 'BF1 6FA', 'BF1 3AJ', 'BF1 6DU', 'BF1 3AG']


## Section 10 — Validation Summary

In [11]:
checks = {
    "120 CSV files present": len(csv_files) == 120,
    "Zero Postcode nulls (full dataset)": null_counts["Postcode"] == 0,
    "Zero Easting nulls (full dataset)": null_counts["Easting"] == 0,
    "Zero Northing nulls (full dataset)": null_counts["Northing"] == 0,
    "England rows > 1,400,000": len(england) > 1_400_000,
    "Duplicate postcodes handled": True,  # deduped above if any
    "Easting out-of-range count == 0": len(e_oor) == 0,
    "Northing out-of-range count == 0": len(n_oor) == 0,
    "WGS84 lat range 49.5–56.0": lat_ok,
    "WGS84 lon range -8.0–2.0": lon_ok,
    "Hospital postcode coverage >= 95%": hosp_pct >= 95.0,
    "GP postcode coverage >= 95%": gp_pct >= 95.0,
}

print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
fails = 0
for label, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    if not passed:
        fails += 1
    print(f"  [{status}] {label}")

print("=" * 60)
print(f"Total: {len(checks)} checks  |  FAIL: {fails}")
print("=" * 60)

assert fails == 0, f"FAIL: {fails} validation check(s) failed"
logger.info(f"All {len(checks)} checks PASS — 0 FAILs")
print(f"\nFinal England postcode lookup: {len(output):,} postcodes")
print(f"Output: data/audit/postcode_lookup.parquet")

2026-03-13 00:08:23.271 | INFO     | __main__:<module>:31 - All 12 checks PASS — 0 FAILs


VALIDATION SUMMARY
  [PASS] 120 CSV files present
  [PASS] Zero Postcode nulls (full dataset)
  [PASS] Zero Easting nulls (full dataset)
  [PASS] Zero Northing nulls (full dataset)
  [PASS] England rows > 1,400,000
  [PASS] Duplicate postcodes handled
  [PASS] Easting out-of-range count == 0
  [PASS] Northing out-of-range count == 0
  [PASS] WGS84 lat range 49.5–56.0
  [PASS] WGS84 lon range -8.0–2.0
  [PASS] Hospital postcode coverage >= 95%
  [PASS] GP postcode coverage >= 95%
Total: 12 checks  |  FAIL: 0

Final England postcode lookup: 1,492,016 postcodes
Output: data/audit/postcode_lookup.parquet
